## Step 1 — Load and pivot the juror votes

Input file is in long format with columns `Voting_country`, `Juror`, `Participating_country`, `Rank`, `DoB`, `ShowOrder`.
Helper functions live in `jury_helpers.py`.

In [1]:
import pandas as pd
from jury_helpers import (
    DICT_ISO, ISO_TO_COUNTRY,
    rank_to_exp_score, rank_to_points,
    rank_jury_with_tiebreakers,
    rank_final_classification,
)

In [2]:
INPUT_FILE = r'jury_votes.xlsx'

long_df = pd.read_excel(INPUT_FILE)

for col in ['Voting_country', 'Participating_country']:
    unknown = set(long_df[col]) - set(ISO_TO_COUNTRY)
    if unknown:
        raise ValueError(f'Unknown ISO codes in {col}: {sorted(unknown)}')
    long_df[col] = long_df[col].map(ISO_TO_COUNTRY)

long_df.head()

,Voting_country,Juror,Participating_country,Rank,DoB,ShowOrder
0,France,Juror 1,United Kingdom,1,2005-12-12,3
1,France,Juror 2,United Kingdom,2,1990-10-24,3
2,France,Juror 3,United Kingdom,2,1963-09-08,3
3,France,Juror 1,Spain,2,2005-12-12,1
4,France,Juror 2,Spain,1,1990-10-24,1


In [3]:
participating_order = long_df['Participating_country'].drop_duplicates().tolist()
voting_order        = long_df['Voting_country'].drop_duplicates().tolist()
juror_order         = long_df['Juror'].drop_duplicates().tolist()

ranks_df = (
    long_df
    .pivot(index='Participating_country',
           columns=['Voting_country', 'Juror'],
           values='Rank')
    .reindex(index=participating_order,
             columns=pd.MultiIndex.from_product(
                 [voting_order, juror_order],
                 names=['Voting_country', 'Juror']))
)
ranks_df

Voting_country         France                 United Kingdom                  \
Juror                 Juror 1 Juror 2 Juror 3        Juror 1 Juror 2 Juror 3   
Participating_country                                                          
United Kingdom              1       2       2              0       0       0   
Spain                       2       1       3              3       1       2   
France                      0       0       0              2       3       1   
Germany                     3       3       1              1       2       3   

Voting_country        Germany                   Spain                  \
Juror                 Juror 1 Juror 2 Juror 3 Juror 1 Juror 2 Juror 3   
Participating_country                                                   
United Kingdom              2       3       1       1       2       2   
Spain                       3       2       2       0       0       0   
France                      1       1       3       2       1       1   
Germany                     0       0       0       3       3       3   

Voting_country        Portugal                  
Juror                  Juror 1 Juror 2 Juror 3  
Participating_country                           
United Kingdom               2       4       3  
Spain                        4       2       1  
France                       3       1       4  
Germany                      1       3       2

In [4]:
juror_dobs = (
    long_df.drop_duplicates(['Voting_country', 'Juror'])
           .set_index(['Voting_country', 'Juror'])['DoB']
)

show_order = (
    long_df.drop_duplicates('Participating_country')
           .set_index('Participating_country')['ShowOrder']
           .to_dict()
)
show_order

{'United Kingdom': 3, 'Spain': 1, 'France': 2, 'Germany': 4}

## Step 2 — Convert ranks to exponential scores

In [5]:
exp_scores_df = ranks_df.map(rank_to_exp_score)
exp_scores_df

Voting_country           France                     United Kingdom            \
Juror                   Juror 1   Juror 2   Juror 3        Juror 1   Juror 2   
Participating_country                                                          
United Kingdom         12.00000   9.92351   9.92351        0.00000   0.00000   
Spain                   9.92351  12.00000   8.20634        8.20634  12.00000   
France                  0.00000   0.00000   0.00000        9.92351   8.20634   
Germany                 8.20634   8.20634  12.00000       12.00000   9.92351   

Voting_country                    Germany                         Spain  \
Juror                   Juror 3   Juror 1   Juror 2   Juror 3   Juror 1   
Participating_country                                                     
United Kingdom          0.00000   9.92351   8.20634  12.00000  12.00000   
Spain                   9.92351   8.20634   9.92351   9.92351   0.00000   
France                 12.00000  12.00000  12.00000   8.20634   9.92351   
Germany                 8.20634   0.00000   0.00000   0.00000   8.20634   

Voting_country                             Portugal                      
Juror                   Juror 2   Juror 3   Juror 1   Juror 2   Juror 3  
Participating_country                                                    
United Kingdom          9.92351   9.92351   9.92351   6.78631   8.20634  
Spain                   0.00000   0.00000   6.78631   9.92351  12.00000  
France                 12.00000  12.00000   8.20634  12.00000   6.78631  
Germany                 8.20634   8.20634  12.00000   8.20634   9.92351

## Step 3 — Sum per jury, rank with tie-breakers, award points

Within-jury tie-breaker chain (when two countries share the same exponential sum):
1. Majority of better individual rankings among that jury's jurors.
2. Vote of the youngest juror.
3. Show of hands — interactive prompt for the winning ISO code.

The cell prints a trace whenever a tie is encountered.

In [6]:
jury_sums = exp_scores_df.T.groupby(level='Voting_country', sort=False).sum().T

jury_sums_for_ranking = jury_sums.copy()
for vc in jury_sums_for_ranking.columns:
    if vc in jury_sums_for_ranking.index:
        jury_sums_for_ranking.loc[vc, vc] = pd.NA

jury_sums

Voting_country,France,United Kingdom,Germany,Spain,Portugal
Participating_country,,,,,
United Kingdom,31.84702,0.00000,30.12985,31.84702,24.91616
Spain,30.12985,30.12985,28.05336,0.00000,28.70982
France,0.00000,30.12985,32.20634,33.92351,26.99265
Germany,28.41268,30.12985,0.00000,24.61902,30.12985


In [7]:
jury_ranks = pd.DataFrame(
    index=jury_sums.index, columns=jury_sums.columns, dtype='Int64'
)

for vc in jury_sums.columns:
    sums = jury_sums_for_ranking[vc]
    ranks_in_jury = ranks_df.xs(vc, axis=1, level='Voting_country')
    dobs_for_jury = juror_dobs.loc[vc].to_dict()
    rank_series = rank_jury_with_tiebreakers(
        vc, sums, ranks_in_jury, dobs_for_jury
    )
    for c, r in rank_series.items():
        jury_ranks.loc[c, vc] = r

jury_ranks

  [United Kingdom jury] Tie at sum=30.12985: ['Spain', 'France', 'Germany']
      majority: France preferred by 2, Spain by 1
      majority: Germany preferred by 2, France by 1
    resolved order: ['Germany', 'France', 'Spain']


Voting_country,France,United Kingdom,Germany,Spain,Portugal
Participating_country,,,,,
United Kingdom,1,<NA>,2,2,4
Spain,2,3,3,<NA>,2
France,<NA>,2,1,1,3
Germany,3,1,<NA>,3,1


In [8]:
jury_points = jury_ranks.map(rank_to_points).astype(int)
jury_points

Voting_country,France,United Kingdom,Germany,Spain,Portugal
Participating_country,,,,,
United Kingdom,12,0,10,10,7
Spain,10,8,8,0,10
France,0,10,12,12,8
Germany,8,12,0,8,12


## Step 4 — Final classification with tie-breakers

Final-tie chain (when two countries have the same total points):
1. Highest number of juries that gave any points.
2. Highest number of 12-point scores.
3. Walk down the points scale — most 10s, then 8s, 7s, 6s, 5s, 4s, 3s, 2s, 1s.
4. Earlier in the show running order wins.

The cell prints a trace whenever a tie is encountered.

In [9]:
final_classification = rank_final_classification(jury_points, show_order)
final_classification

,Total_points,Rank
Participating_country,,
France,42,1
Germany,40,2
United Kingdom,39,3
Spain,36,4
